In [1]:
import pandas as pd
import scanpy as sc

In [2]:
BASE_DIR = 'D:/Gdrive/notebook/2509_scrnaseq_deconv/260123_puseudo_data'
import sys
sys.path.append(f'{BASE_DIR}')
import simulation

In [3]:
adata = sc.read_h5ad(f'{BASE_DIR}/result_32699x3000.h5ad')
adata

AnnData object with n_obs × n_vars = 32699 × 3000
    obs: 'pct_counts_mt', 'n_genes_by_counts', 'predicted_doublet', 'log1p_total_counts_hb', 'pct_counts_in_top_50_genes', 'doublet_score', 'total_counts', 'total_counts_ribo', 'pct_counts_hb', 'pct_counts_in_top_100_genes', 'n_genes', 'pct_counts_in_top_500_genes', 'pct_counts_in_top_200_genes', 'total_counts_mt', 'log1p_n_genes_by_counts', 'total_counts_hb', 'log1p_total_counts_ribo', 'cell_type', 'pct_counts_ribo', 'log1p_total_counts', 'log1p_total_counts_mt', 'origin', '_scvi_batch'
    var: 'n_cells_by_counts', 'ribo', 'pct_dropout_by_counts', 'log1p_mean_counts', 'mean_counts', 'mt', 'log1p_total_counts', 'hb', 'n_cells', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'mean', 'std'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'cell_type_colors', 'hvg', 'log1p', 'neighbors', 'origin_colors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    var

In [4]:
dat = simulation.Liver_Simulator(adata, sample_size=8000, method='dirichlet')

In [5]:
dat.assign(group_weights=[0.7, 0.15, 0.15]) # [parenchymal, supporting, immune]
summary_df = dat.summary_df
summary_df.to_csv(f'{BASE_DIR}/data/gs_8000x11_assign.csv')

summary_df.head()

,Hepatocytes,Cholangiocytes,Fibroblasts,Neutrophils,Macrophages,Monocytes,NK,CD4Tcells,CD8Tcells,B cells,Dendritic
0,0.7,0.040006,0.109994,0.011244,0.025735,0.040860,0.014070,0.001692,0.016246,0.002126,0.038026
1,0.7,0.076379,0.073621,0.000289,0.001090,0.005639,0.021298,0.055607,0.013505,0.039247,0.013326
2,0.7,0.102515,0.047485,0.000706,0.017959,0.042011,0.047538,0.006969,0.024958,0.001981,0.007880
3,0.7,0.069747,0.080253,0.003602,0.040908,0.025945,0.002423,0.039567,0.010515,0.013492,0.013549
4,0.7,0.066089,0.083911,0.000191,0.003776,0.028851,0.009470,0.038945,0.009784,0.056647,0.002335


In [6]:
dat.assign() # 指定がないときは[parenchymal, supporting, immune]もdirichletに従う
summary_df = dat.summary_df
summary_df.to_csv(f'{BASE_DIR}/data/gs_8000x11_dirichlet.csv')

summary_df.head()

,Hepatocytes,Cholangiocytes,Fibroblasts,Neutrophils,Macrophages,Monocytes,NK,CD4Tcells,CD8Tcells,B cells,Dendritic
0,0.244940,0.002885,0.049492,0.013900,0.067660,0.089565,0.062227,0.055603,0.237563,0.140738,0.035425
1,0.171006,0.257404,0.043445,0.139427,0.029343,0.104903,0.082048,0.065414,0.055695,0.020540,0.030775
2,0.085523,0.354929,0.273092,0.026478,0.037303,0.033901,0.006837,0.002247,0.118405,0.004308,0.056977
3,0.352658,0.264664,0.066902,0.038271,0.014798,0.011719,0.011370,0.073608,0.034787,0.087744,0.043478
4,0.043884,0.073397,0.116338,0.006522,0.161917,0.006447,0.019260,0.334249,0.011252,0.006155,0.220578


In [7]:
dat.split_cell_idx(save_dir=f'{BASE_DIR}/data/cell_idx', train_ratio=0.7)
cell_idx_dict = dat.cell_idx_dict
pd.to_pickle(cell_idx_dict, f'{BASE_DIR}/data/cell_idx/cell_idx_dict.pkl')

Hepatocytes: 10625 cells detected
Cholangiocytes: 647 cells detected
Fibroblasts: 1260 cells detected
Neutrophils: 5339 cells detected
Macrophages: 7067 cells detected
Monocytes: 1369 cells detected
NK: 334 cells detected
CD4Tcells: 2136 cells detected
CD8Tcells: 756 cells detected
B cells: 3035 cells detected
Dendritic: 70 cells detected


In [8]:
summary_df = summary_df.iloc[0:10,:] # 時短で10サンプルの作成
dat.set_data(summary_df=summary_df) # summary_dfだけ時短verをset

bulk_df = dat.create_sim_bulk(pool_size=500, mode='train')
bulk_df.to_csv(f'{BASE_DIR}/data/bulk_3000_10_train.csv')

bulk_df.head()

Log-transformation detected. Reverting to linear scale...
Data check: Negative values clipped to 0. This adata contains count data.


100%|██████████| 10/10 [00:00<00:00, 204.04it/s]


,0,1,2,3,4,5,6,7,8,9
7SK.2,372.321621,274.919556,154.288396,484.058385,428.487801,461.798136,782.481938,522.754583,531.904740,662.607529
AAGAB,347.125121,516.899965,570.922769,751.901603,202.265053,771.198768,879.922933,736.233072,706.884729,789.283771
AATK,158.981838,501.853412,326.247796,359.345266,1586.052911,228.263249,45.150122,791.403253,406.415199,473.886044
ABCA7,315.303190,410.187588,176.336071,248.921437,133.121657,169.084891,175.902743,336.420861,147.432914,166.570552
ABCC12,458.718421,427.866436,552.985245,630.133124,701.412090,1003.686266,957.508561,542.851167,760.639186,993.911902


In [9]:
bulk_df = dat.create_sim_bulk(pool_size=500, mode='test')
bulk_df.to_csv(f'{BASE_DIR}/data/bulk_3000_10_test.csv')

bulk_df.head()

Data check: Negative values clipped to 0. This adata contains count data.


100%|██████████| 10/10 [00:00<00:00, 249.95it/s]


,0,1,2,3,4,5,6,7,8,9
7SK.2,369.774663,293.831547,138.528587,529.701089,442.053869,460.491647,784.650639,522.540089,540.538352,585.710669
AAGAB,333.681534,515.757116,515.279292,766.958807,179.714029,701.679276,954.150420,616.917929,677.137891,879.312409
AATK,193.323637,731.886139,304.337600,391.426566,1184.836324,152.411710,481.349877,359.164092,109.212992,403.686720
ABCA7,319.523222,548.320306,163.023153,286.659920,99.165085,183.882435,177.167886,314.071210,158.053449,168.074768
ABCC12,551.080614,500.435125,598.183893,801.514412,608.756552,1065.319217,874.824647,801.671231,1036.622166,884.174246
